**Notebook to Calibrate Wires trajectory for DAXM measurements from Fluorescence intensity shadowing**

- load a DAXM scan with visible shadows lines (scan performed on single sample point that produces fluorescence)
- GUI to find a set approached values for the N wires
- Refinement of Calibration parameters

**Section 2**: to start from scratch and select a daxm scan of interest

**Section 3**: GUI to find rather good parameters of the wires trajectory and positions. A .scan file is saved with these parameters in a created analysis directory. By default it is called `init_calib.scan`
This section can be skipped

**Section 4**: Calibration procedure, which can be started by loading `init_calib.scan` file.

#-------------------------------------------------------------

Initial version: Robin Freville based on Loic Renversade Code

Major Revision by Clement <user> 2026

Last Revision by J.-S. Micha June 2026

# IMPORTS AND ENVIRONMENTS

In [ ]:
!which python

In [ ]:
import os, sys, ipympl
print(f'you are using pythn {sys.version}')
if sys.version_info.minor>=8:
    %matplotlib ipympl
    try:
        import ipympl
    except:
        print('You need to install ipympl: pip install ipympl')
else:
    %matplotlib notebook

In [ ]:
sys.executable, sys.version

In [ ]:
import os, time, copy
import itertools
import glob
from pathlib import Path

import matplotlib.pyplot as plt
#----------- global imports -----------

import matplotlib as mpl
import matplotlib.pylab as mplp
import matplotlib.pyplot as plt

import numpy as np

from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
if 0: # test if we can get some plots (depending on notebook version ... and ipympl needs or not)
    fig_,ax_ = plt.subplots()
    ax_.plot(np.arange(20))

In [ ]:
# in jupyter-slurm for venv environment only (not conda !!  for this you need to create a ipykernel and)
# LaueToolsCode_Folder = '<inhouse path>'
# sys.path.insert(0,LaueToolsCode_Folder)
# print(sys.path)
# import LaueTools as LT
# print('\ncode from', LT.__file__)

In [ ]:
import fabio
import pandas as pd
from pathlib import Path
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D

In [ ]:
#----------- daxm imports -----------
from LaueTools.Daxm.classes.scan import new_scan
from LaueTools.Daxm.classes.source import new_source
#from LaueTools.Daxm.classes.calibration import CalibManager
#from LaueTools.Daxm.classes.reconstruction import SpotReconstructor
from LaueTools.Daxm.classes.scan.point import StaticPointScan, new_scan_dict, PointScan, save_scan_dict
import LaueTools.generaltools as GT
import LaueTools.logfile_reader as iohdf5
import LaueTools.blissdatafolderstructure as blissf
import LaueTools.daxm_workflow as DW   # fast list of the wire scans, scan parameters
import LaueTools.Daxm.quickdaxm as QD   # image reader (single or stacked images files)
GT.__file__

In [ ]:
#print(mpl.get_backend())
#sys.version

In [ ]:
if 0: # test if we can get some plots (depending on notebook version ... and ipympl needs or not)
    fig_,ax_ = plt.subplots()
    ax_.plot(np.arange(20))

# SET DAXM SCAN, DETECTOR, USER FOLDER

In [ ]:
# example of the way data are organised in folders tree
# sample='Au15_1'
# dataset = '2Dmap_after'
# f'{sample}_{dataset}'
# f'/data/visitor/<proposal>/bm32/<date>/RAW_DATA/{sample}/{sample}_{dataset}'

configurations = {
"<proposal>": {
        "maindatafolder_RAW_DATA": "/data/visitor/<proposal>/bm32/<date>/RAW_DATA/",
        "mainfolder_PROCESSED_DATA": "/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": "<sample>",
        "NewDataset_Name": "daxmfluo",
        "Scan_repo_Nb": "0001",
    },
"<proposal>": {
        "maindatafolder_RAW_DATA": "/data/visitor/<proposal>/bm32/<date>/RAW_DATA",
        "mainfolder_PROCESSED_DATA": "/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA",
        "CCDType": "sCMOS_4M",
        "Newsample_Name": "<sample>_R3",
        "Zone": "Z1",
        "NewDataset_Name": "Z1_daxm_dia_combo_Line_1_10umstep",
        "Scan_repo_Nb": "0001",
    },

"<proposal>": {
        "maindatafolder_RAW_DATA": "/data/visitor/<proposal>/bm32/<date>/RAW_DATA",
        "mainfolder_PROCESSED_DATA": "/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": "<sample>",
        "Zone": None,
        "NewDataset_Name": "gedaxm",
        "Scan_repo_Nb": "0001",
    },

"<proposal>": {
        "maindatafolder_RAW_DATA": "/data/visitor/<proposal>/bm32/<date>/RAW_DATA",
        "mainfolder_PROCESSED_DATA": "/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": "<sample>",
        "Zone": None,
        "NewDataset_Name": "aligndaxm",
        "Scan_repo_Nb": "0042",
    },
"<proposal>": {
        "maindatafolder_RAW_DATA": "/data/visitor/<proposal>/bm32/<date>/RAW_DATA",
        "mainfolder_PROCESSED_DATA": "/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": "<sample>",
        "Zone": None,
        "NewDataset_Name": "Gedaxm",
        "Scan_repo_Nb": "0006",  # 501 pts
    },
    "<proposal>": {
        "maindatafolder_RAW_DATA": "/data/visitor/<proposal>/bm32/<date>/RAW_DATA/",
        "mainfolder_PROCESSED_DATA": "/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA",
        "CCDType": "EIGER_4MCdTe",
        "Newsample_Name": None,
        "Zone": None,
        "NewDataset_Name": None,
        "Scan_repo_Nb": None,  
    }
}


In [ ]:
# Select the configuration
expId = "<proposal>"
expId ="<proposal>"

# =================================================
config = configurations[expId]
GT.printgreen(f'selected config :\n {config}\n')
#Unpack the dictionary into variables
locals().update(config)
CCDLabel = CCDType

# Now you can use the variables directly:
print('Variables ----')
print('maindatafolder_RAW_DATA: ',maindatafolder_RAW_DATA)
print('CCDType: ',CCDType)
#print('NewDataset_Name: ',NewDataset_Name)

# check if images data folder exists:
imagefolder = Path(maindatafolder_RAW_DATA)/Newsample_Name/f'{Newsample_Name}_{NewDataset_Name}'/f'scan{Scan_repo_Nb}'
if not Path.exists(imagefolder):
    GT.printred(f'Folder {imagefolder} does not exist !!')
else:
    GT.printgreen(f'Folder {imagefolder} exists !!')

## Find automatically master .h5 file or SET local .h5 file or 

In [ ]:
mainh5file, HDF5_LOGFILE_EXISTS = blissf.findmasterh5file(maindatafolder_RAW_DATA)

print('mainh5file',mainh5file)

if mainh5file:
    GT.printgreen(f'Found master .h5 file')
    h5file = str(Path(f"{maindatafolder_RAW_DATA}")/mainh5file)
    GT.printgreen(f'set `h5file` to  : {h5file}')
else:
    GT.printyellow('H5 file not found! Check carefully the folder')
    HDF5_LOGFILE_EXISTS = False
    

In [ ]:
blissf.tree(maindatafolder_RAW_DATA,0)

In [ ]:
blissf.tree(Path(maindatafolder_RAW_DATA)/Newsample_Name,0, truncatesize=15)

In [ ]:
# top level master h5 file
h5file, HDF5_LOGFILE_EXISTS = blissf.findmasterh5file(Path(maindatafolder_RAW_DATA))
h5file

## Read the hdf5 files: list of the wire scans (fast)

The wire scans (`ascan`, `fscan`, `dscan` of `zf` or `yf`) are read directly in the hdf5 files of the datasets (`RAW_DATA/<sample>/<dataset>/<dataset>.h5`), each file opened once and in parallel: a few seconds, even for an experiment with thousands of scans. The master hdf5 file of the experiment (`h5file`, slow for large experiments) is not needed.

- `SCAN_FILTER`: words of the dataset name (case insensitive); a word starting with `-` must not be in it, e.g. `'daxm -grid'`, `'Ge'`; `None`: all the wire scans
- columns: `dataset` (name of the scan as in the master file), `points` (recorded wire positions), `image_files` (files in the scan folder: 1 for all the images stacked in one file), `end` (end reason of the scan)

In [ ]:
SCAN_FILTER = '<sample>'

wirescans = DW.list_wire_scans(maindatafolder_RAW_DATA, contains=SCAN_FILTER)

# [FORMER, SLOW] reading through the master hdf5 file of the experiment (about 30 s for 5000 scans)
USE_MASTER_FILE = False
if USE_MASTER_FILE and HDF5_LOGFILE_EXISTS:
    logfile = iohdf5.H5file(h5file, CCDType)
    print('selected logfile .h5 :', logfile.path)
    logfile.getscans(verbose=1, n_last=10000, exclude_substrings=['grid'])
    dfallscans = logfile.dfallscans
    dfzfscans = logfile.filterby('motors', 'zf', openGUI=False)

In [ ]:
wirescans[['dataset', 'command', 'points', 'image_files', 'end', 'start']]

In [ ]:
# e.g. Ge wire scans only
# wirescans[wirescans['dataset'].str.contains('Ge', case=False)]

## SELECT scan of interest

set `DAXM_SCAN` to the dataset name of the scan (column `dataset`) or to its index (left column of the table above)

In [ ]:
DAXM_SCAN = '<dataset>_1'

wirescan = DW.select_wire_scan(wirescans, DAXM_SCAN)
# images files, detector label (EIGER_4MCdTestack if all the images are in one file), wire positions...
scanparams = DW.scan_parameters_from_hdf5(wirescan['hdf5_file'], wirescan['scan'],
                                          CCDLabel=None if CCDType.startswith('EIGER') else CCDType)
print(wirescan['command'], f"({wirescan['end']})")
scanparams

## SELECT a DAXM scan and SET parameters

it will create a StaticPointScan object that will be saved further in a ####_calib.scan

In [ ]:
# uncomment to be helped to browse over the good path
#!ls '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0001/calibGe001_eiger4m.det'

In [ ]:
# detector calibration file and number of wires of each experiment
# (calibration files of an experiment, most recent first: import LaueTools.map_workflow as MW; MW.find_detfiles(maindatafolder_RAW_DATA))
experiment_settings = {
    '<proposal>': {'number_wire': 3,
        'calibdet_file': '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0002/calibGe001_Eiger_June2026_dist100mm.det'},
    '<proposal>': {'number_wire': 3,
        'calibdet_file': '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0001/calibGe001_eiger4m.det'},
    '<proposal>': {'number_wire': 3,
        'calibdet_file': '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0006/calibGe001_eiger4m_zcam100mm.det'},
    '<proposal>': {'number_wire': 4,
        'calibdet_file': '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0004/calibGe001_eiger4m_70mm_Ysmaller1500.det'},
    '<proposal>': {'number_wire': 4, 'calibdet_file': f'{maindatafolder_RAW_DATA}/Ge1/Ge1_calib_std/scan0001/Ge1.det'},
    '<proposal>': {'number_wire': 4,
        'calibdet_file': '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/scan0001/CalibGe001.det'},
}
number_wire = experiment_settings[expId]['number_wire']
calibdet_file = experiment_settings[expId]['calibdet_file']
# calibdet_file = [80, 1079.6100, 983.0300, 0.2030000, 0.3220000, 0.07500000]  # or the 6 parameters (fake distance)

#==================== scan selected in section 2.2 (scanparams) =================================================
image_folder = scanparams['image_folder']
prefix = scanparams['prefix']
suffix = scanparams['suffix']
sizeofzeropadding = scanparams['nbdigits']
localhdf5file = scanparams['hdf5_file']

raw_data_index = Path(image_folder).parts.index('RAW_DATA')
# The next two parts after 'RAW_DATA' are the first and second subfolders
Newsample_Name = Path(image_folder).parts[raw_data_index + 1]
NewDataset_Name = Path(image_folder).parts[raw_data_index + 2]
Scan_repo_Nb = Path(image_folder).parts[raw_data_index + 3]
analysis_dir = f"{mainfolder_PROCESSED_DATA}/{Newsample_Name}/{NewDataset_Name}/{Scan_repo_Nb}"  ## Folder to write the _calib.scan file

# Make sure directory exists
os.makedirs(analysis_dir, exist_ok=True)
GT.printgreen(f'analysis_dir : {analysis_dir}')

scan_dict = new_scan_dict()

fullcommand = wirescan['command']
scantype = fullcommand.split()[0]
scan_number = int(wirescan['scan'])
ScanID = wirescan['dataset']   # <dataset>_<scan number>, name of the scan in the master hdf5 file

scan_dict["CCDType"] = scanparams['CCDLabel']
scan_dict["specFile"] = h5file if HDF5_LOGFILE_EXISTS else localhdf5file
scan_dict["scanNumber"] = scan_number
scan_dict['hdf5scanId'] = ScanID  #  hdf5 index
scan_dict["imageFolder"] = image_folder
scan_dict["imagePrefix"] = prefix
scan_dict["imageFirstIndex"] = scanparams['first_image_index']
scan_dict["imageDigits"] = sizeofzeropadding
scan_dict["detCalib"] = calibdet_file
scan_dict['wire'] = [['W', 0.025, 0.7, 8] for k in range(number_wire)]
scan_dict["scantype"] = scantype
scan_dict["fullcommand"] = fullcommand
scan_dict["localhdf5file"] = localhdf5file
if CCDType.startswith('EIGER'):
    scan_dict['imageOffset'] = 0 # pedestal of image intensity signal
    scan_dict["monitorOffset"] = 31.7 # MArch 2026
    scan_dict["monitorOffset"] = 28.3 # June 2026
else:
    scan_dict['imageOffset'] = 1000 # pedestal of image intensity signal
    scan_dict["monitorOffset"] = 25  # old data

scantype, scan_dict

In [ ]:
# example how to get data from a given scan
# import h5py

# with h5py.File(localhdf5file, 'r', locking=False) as f:
#     scannode = f[f'{scan_dict["scanNumber"]}.1']
#     data = scannode['measurement']
#     data2 = scannode['measurement/mon'][()]
#     wirep = scannode[f'instrument/positioners/zf'][()]
#     # print(data['mon'])
#     # print(data['mon'][()])
#     data2

In [ ]:
## Create the scan ## 
scan = StaticPointScan(scan_dict, verbose=False)

In [ ]:
#scan.calc_wires_range_shadow(300)

In [ ]:

print('approx. p0 value', scan.wire_position[0], scan.wire_position[-1])
print('Value to put in the next simulator GUI')
estimatedp0 = 0.5*(scan.wire_position[0]+ scan.wire_position[-1])
print(np.round(estimatedp0,2))


# SIMULATION OF WIRES SHADOWS

It can be skipped if you have already a ####_calib.scan ready for refinement

- The GUI will help to find the initial wires trajectory parameters before fitting
- a corresoponding ####_calib.scan will be produced

In [ ]:
def ShowWireShadows(imageindex, ysrc, ax, selected=None):
    """draw the shadow of each wire on the detector for a source at ysrc (microns) along the beam.
    selected: index (from 0) of the wire edited in the boxes, drawn more opaque"""

    xmin, xmax = 0, scan.get_img_params(['framedim'])[0]
    ys_right = scan.calc_wires_range_shadow(imageindex , xcam=xmax, ysrc=ysrc/1000.)
    ys_left = scan.calc_wires_range_shadow(imageindex, xcam=xmin, ysrc=ysrc/1000.)

    align = 'top'

    colors = mplp.rcParams['axes.prop_cycle'].by_key()['color']
    
    for i, ymin in enumerate(ys_left):
        ymax = ys_right[i]
        line = Line2D([xmin, xmax], [ymin[0], ymax[0]],
                      antialiased=True, color=colors[i], linestyle='-')
        poly = Polygon(list(zip([xmin, xmax, xmax, xmin],
                           [ymin[1], ymax[1], ymax[2], ymin[2]])),
                       closed=True, antialiased=True, alpha=0.6 if i == selected else 0.25, facecolor=colors[i])

        txt = ax.text(xmax-10, ymin[1], str(i+1) + (' (edited)' if i == selected else ''), fontsize='small',
                                   horizontalalignment='right', verticalalignment=align)

        ax.add_line(line)
        ax.add_patch(poly)


# True while the boxes are filled with the parameters of the selected wire: the boxes observers must not
# modify any wire then (filling the boxes one by one would mix the parameters of two wires)
_filling_boxes = False


def redraw():
    update_img(imageindex=imageindex_widget.value, showWire=showWire_widget.value, wire_selector=wire_selector.value,
               vmax=vmaxslider.value, ysrc=ysrc_value.value)


def update_wire(change=None):
    """set the parameters of the selected wire from the boxes (angles in degrees in the boxes, radians in scan)"""
    if _filling_boxes:
        return
    scan.set_wire(wire_selector.value - 1, {'R': float(R_value.value),
                                            'h': float(h_value.value),
                                            'p0': float(p0_value.value),
                                            'f1': np.deg2rad(float(f1_value.value)),
                                            'f2': np.deg2rad(float(f2_value.value))}, verbose=0)
    if showWire_widget.value:
        redraw()
    else:
        showWire_widget.value = True   # shows the shadows (redraws the image)


def reset_box(change):
    """fill the boxes with the parameters of the wire selected with the slider (no wire is modified)"""
    global _filling_boxes
    wire = scan.get_wire_dict(int(change['new']) - 1)
    _filling_boxes = True
    try:
        h_value.value = float(wire['h'])
        p0_value.value = float(wire['p0'])
        f1_value.value = float(np.rad2deg(wire['f1']))
        f2_value.value = float(np.rad2deg(wire['f2']))
        R_value.value = float(wire['R'])
    finally:
        _filling_boxes = False

In [ ]:
## WIDGET DEFINITION TO CALIBRATE WIRE ##
h_value = widgets.FloatText(
    value=0.7,
    description='h:',
    disabled=False
)

p0_value = widgets.FloatText(
    value=estimatedp0,
    description='p0:',
    disabled=False
)

f2_value = widgets.FloatText(
    value=0.,
    description='f2 (°):',
    disabled=False
)

f1_value = widgets.FloatText(
    value=0.,
    description='f1 (°):',
    disabled=False
)

R_value = widgets.FloatText(
    value=0.025,
    description='R:',
    disabled=False
)

ysrc_value = widgets.FloatText(
    value=0,
    description='ysource (µm)',
    disabled=False
)

vbox = widgets.VBox([h_value, p0_value, f1_value, f2_value, R_value, ysrc_value])

imageindexmax = scan.number_images-1
#imageindexmax = 800 # for simulation

imageindex_widget = widgets.IntSlider(value=0, min=0, max=imageindexmax, description="Image index")
showWire_widget = widgets.Checkbox(value=False, description="Show wire")
wire_selector = widgets.IntSlider(value=1, min=1, max=scan.wire_qty, description="Wire selector")
vmaxslider = widgets.IntSlider(value=2000, min=10, max=6000, description="vmax")
# source position along the beam (microns): one control (ysrc_value in the boxes) for the image and the shadows

ui = widgets.VBox([vmaxslider, imageindex_widget, showWire_widget, wire_selector])

# wires whose p0 is outside the wire scan start at the middle of the scan (as the p0 box)
for _iw in range(scan.wire_qty):
    if not min(scan.wire_position[0], scan.wire_position[-1]) <= scan.get_wire_dict(_iw)['p0'] <= max(scan.wire_position[0], scan.wire_position[-1]):
        scan.set_wire(_iw, {'p0': float(estimatedp0)}, verbose=0)
# boxes = parameters of the selected wire (wire 1)
reset_box({'new': wire_selector.value})

In [ ]:
scan.number_images

## GUI VISUALISATION of WIRES shadows

In [ ]:
figplot, ax = mplp.subplots()
ax.set_xlim(0,scan.get_img_params(['framedim'])[0])
ax.set_ylim(scan.get_img_params(['framedim'])[1],0)


def update_img(imageindex=0, showWire = False, wire_selector=1, vmax=2000, ysrc=0):
    ymin, ymax = ax.get_ylim()
    xmin, xmax = ax.get_xlim()
    ax.clear()
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_xlabel("Xcam")
    ax.set_ylabel("Ycam")
    if imageindex>=scan.number_images:
        imageindex = scan.number_images-1
        #print('no image available')
    # one image per file or all the images stacked in one file (EIGER)
    img = QD.read_image(scanparams, imageindex)
    c1 = int(np.mean(img, axis=(0, 1))) + 4 * int(np.std(img, axis=(0, 1)))
    #vmin, vmax = 1000, int(c1)
    vmin = 0
    
    ax.imshow(img, origin='upper', vmin=vmin, vmax=vmax, cmap='grey')
    if showWire:
        ShowWireShadows(imageindex, ysrc, ax, selected=wire_selector - 1)

    plt.show()

        
out = widgets.interactive_output(update_img, {
    "imageindex": imageindex_widget,
    "showWire": showWire_widget,
    "wire_selector": wire_selector,
    "vmax":vmaxslider,
    "ysrc": ysrc_value
})

# slider: the boxes show the parameters of the selected wire (no wire modified)
wire_selector.observe(reset_box, names='value')
# boxes: modify the selected wire only
for _box in (h_value, p0_value, f1_value, f2_value, R_value):
    _box.observe(update_wire, names='value')
display(ui, out)
display(vbox)

## SEE the wires parameters

WARNING, f1 and f2 (tilt angles of the wires, mostly f1 is visible) are given here in radians in scan.get_wires_dict() (in degrees in the boxes of the GUI above)

In [ ]:
scan.get_wires_dict()

## SET filename and SAVE a ####_calib.scan

In [ ]:
ScanID, analysis_dir, scan.scantype

In [ ]:
#ScanID: Corresponds to 'prefix' in 'header.py'
#prefix_DAXM_analysis = ScanID
prefix_DAXM_analysis = 'init'

SavedName = f'{prefix_DAXM_analysis}_calib.scan'

#=====================================================================
scan.input['wire'] = scan.get_wires_dict()

# Take too much time  !!!
point_scan = PointScan(scan.input, verbose=False)
point_scan.save(SavedName, directory=analysis_dir)
GT.printgreen(f'\n####_calib.scan file saved in {os.path.join(analysis_dir,SavedName)}')

In [ ]:
point_scan.scantype, point_scan.wire_step, point_scan.number_images, point_scan.wire_params

In [ ]:
point_scan.localhdf5file

# CALIBRATION wires trajectory parameters refinement

- use the initial parameters found above

- or restart directly from a .scan file  (either init_calib.scan or some .scan file after refinement

Following the calib.py script

In [ ]:
analysis_dir = '/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA/<sample>/<dataset>/scan0001'
analysis_dir

In [ ]:
import os
import matplotlib.pylab as mplp
from LaueTools.Daxm.classes.scan.scan import new_scan
from LaueTools.Daxm.classes.source import new_source
from LaueTools.Daxm.classes.calibration import new_calib, CalibManager
from LaueTools.Daxm.classes.reconstruction import RecManager
print("daxm classes loaded ...")

## SET Output Folders: reconstruction and calibration

In [ ]:
from pathlib import Path

calib_dir = Path(analysis_dir+'/'+'calibration')
rec_dir = Path(analysis_dir+'/'+'reconstruction')

# Make sure directory exists
os.makedirs(calib_dir, exist_ok=True)
# Make sure directory exists
os.makedirs(rec_dir, exist_ok=True)

print(' "calib_dir" set to', calib_dir)
print(' "rec_dir" set to', rec_dir)

the following comes from script calib.py

In [ ]:
# look at already saved .scan file
!ls {analysis_dir}

In [ ]:
# edit a scan file to check
!cat {Path(analysis_dir)/'init_calib.scan'}

In [ ]:
# TODO !!!!!   read all scans !!

#SavedName ='<sample>_aligndaxm_42_calib.scan'
#SavedName ='<dataset>_6_calib.scan'
#SavedName = '<sample>_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_calib.scan'
SavedName = 'init_calib.scan'

# previousscanfile_for_calib = '<sample>_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_calib.scan'
# calib_file = os.path.join(analysis_dir, previousscanfile_for_calib)

calib_file = os.path.join(analysis_dir, SavedName)


#----------- load/create scan, sample, calib -----------
scan = new_scan(calib_file, verbose=False)
ScanID = scan.hdf5scanId

In [ ]:
#scan.get_wires_dict()

In [ ]:
sample = new_source("Ti", 0.5, ystep=0.001)
calib = new_calib(scan, sample, kind="fluo")

In [ ]:
# automatic selection of grid points on detector

#TODO  !!!!   repair adjust = True
# TODO  !!!!  accelerate Loading (and calculating ?) experimental intensity profiles ...

calib.set_points_grid(dims=[2, 2], adjust=False)


In [ ]:
# manual selection of points on detector
# needs to launch anyway calib.set_points_grid(dims=[2, 2], adjust=False) to init other things (nb of points ....)
# will set calib.data_XYcam

listptsW1=[[689,1280],[1308,1231],[352,1201],[1474,1365]]
listptsW2=[[689,822],[1378,822],[689,1011],[1350,950]]
listptsW3=[[689,182],[1378,182],[689,365],[1385,365]]
XYcam = [listptsW1, listptsW2, listptsW3]
calib.set_points(XYcam)

In [ ]:
wire_index = 2
calib.plot_exp_wire(wire_index)

In [ ]:
#----------- coarse calibration-----------

#bounds=([-.1,.1],[-.2,.2],[-1,1])  # degrees for axis or f1 f2 u1 u2, and SHOULD be diffeentila value around initial guess 
#calib.run(var=['h', 'p0', 'f1'])#, bounds=bounds)
# calib.run(var=['h', 'p0','axis','dm'])
# intermediatecalibfile = ScanID+"_notebook_coarse"
# calib.save_wires(intermediatecalibfile, directory=calib_dir)
# calib.log_plot()
# mplp.show()



# #-------------------------------------------
# calib.load_wires(ScanID+"_notebook_coarse_radii", directory=calib_dir, verbose=3)
# calib.log_restart()



# #calib.load_wires(intermediatecalibfile, directory=calib_dir)
# #calib.set_points_grid(dims=[2, 2], adjust=False)
# calib.run(var=['h', 'p0','R'])

# intermediatecalibfile_1 = ScanID+"_notebook_coarse_radii2"
# calib.save_wires(intermediatecalibfile_1, directory=calib_dir)
# calib.log_plot()
# mplp.show()

# ###---------------------------------------------------------
# #intermediatecalibfile_1 = '<sample>_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_notebook_coarse_radii'

# #----------- coarse calibration-----------
# calib.load_wires(intermediatecalibfile_1,directory=calib_dir, verbose=3)
# calib.log_restart()
# calib.set_points_grid(dims=[3, 3], adjust=False)
# calib.run(var=['h', 'p0', 'axis', 'R', 'u2', 'dm'], verbose=3)
# intermediatecalibfile2a = ScanID+"_notebook_lesscoarse"
# calib.save_wires(intermediatecalibfile2a, directory=calib_dir)
# calib.log_plot()
# mplp.show()

#----------- coarse calibration-----------
calib.load_wires(intermediatecalibfile2a,directory=calib_dir, verbose=3)
calib.log_restart()
calib.set_points_grid(dims=[3, 3], adjust=False)
calib.run(var=['h', 'p0', 'axis', 'R', 'u1', 'u2', 'dm'], verbose=3)
intermediatecalibfile2b = ScanID+"_notebook_lesscoarse"
calib.save_wires(intermediatecalibfile2b, directory=calib_dir)
calib.log_plot()
mplp.show()

#----------- fine calibration-----------

calib.load_wires(intermediatecalibfile2b, directory=calib_dir)
calib.log_restart()
calib.data_span = 3.5
calib.set_points_grid(dims=[6, 6],adjust=False)
calib.run(var=['R', 'h', 'p0', 'axis', 'u1', 'u2', 'dm'])
finalcalibfile = ScanID+"_notebook"
calib.save_wires(finalcalibfile, directory=calib_dir)
calib.log_plot()
mplp.show()


In [ ]:
# calib.log_plot()
# mplp.show()

In [ ]:
#<sample>_Gedaxm_expo_0.2_nbsteps_400_hmicro1mm_1_calib

In [ ]:
calib.load_wires(ScanID+"_notebook", directory=calib_dir)
calib.log_restart()
calib.data_span = 3.5
calib.set_points_grid(dims=[6, 6],adjust=False)
calib.run(var=['R', 'h', 'p0', 'axis', 'u1', 'u2', 'dm'])
finalcalibfile = ScanID+"_notebook_allparams"
calib.save_wires(finalcalibfile, directory=calib_dir)
calib.log_plot()
mplp.show()

In [ ]:
calib.data_XYcam

# Test performance

In [ ]:
import cProfile
import pstats
import numpy as np

def profile_code():
    # Your code here
    XYcam = np.concatenate(calib.data_XYcam)
    wire = np.concatenate([[i] * len(xy) for i, xy in enumerate(calib.data_XYcam)])
    data_I, data_pw = calib.scan.get_profile_manypixels_centred(wire, XYcam, calib.data_hbs, calib.data_span)

    for i, wid in enumerate(wire):
        calib.data_pw[wid].append(data_pw[i])
        calib.data_I[wid].append(data_I[i])

# Run the profiler
cProfile.runctx('profile_code()', globals(), locals(), 'profile_stats')

# Print the profiling results
stats = pstats.Stats('profile_stats')
stats.strip_dirs().sort_stats('cumtime').print_stats(10)  # Show top 10 time-consuming functions